# Tool Calling

In [ ]:
import openai
import os

openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI

weather_tool = {
    "type": "function",
    "name": "get_current_weather",
    "description": "Get the current weather in a given location",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
        },
        "required": ["location"],
    },
}

In [ ]:
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    # For the sake of this example, we're just going to return a fixed response
    return {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }

In [ ]:
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o",
    tools=[weather_tool],
    tool_choice="auto",
    messages=[
        {
            "role": "user",
            "content": "What's the weather like in Boston?",
        }
    ],
)
print(response.choices[0].message)
# If the model decided to call the tool, we can execute it and send the results back to the model
if response.choices[0].message.tool_calls:
    tool_call = response.choices[0].message.tool_calls[0]
    if tool_call["name"] == "get_current_weather":
        tool_args = tool_call["arguments"]
        tool_response = get_current_weather(**tool_args)
        second_response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": "What's the weather like in Boston?",
                },
                {
                    "role": "assistant",
                    "content": None,
                    "tool_calls": [
                        {
                            "name": "get_current_weather",
                            "arguments": tool_args,
                            "response": tool_response,
                        }
                    ],
                },
            ],
        )
        print(second_response.choices[0].message.content)

# AI Agents using LangChain

In [ ]:
import os
from langchain_core.tools import tool
from langchain_core.runnaables import RunnablePassThrough
from langchain_core.output_parsers import StructuredOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
loader = TextLoader("state_of_the_union.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_name="state_of_the_union",
)
retriever = vectorstore.as_retriever()
rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that answers questions about the US State of the Union address."),
        ("human", "{question}"),
        ("tool", "{retrieved_docs}"),
    ]
)

def format_retrieved_docs(retrieved_docs):
    formatted_docs = ""
    for i, doc in enumerate(retrieved_docs):
        formatted_docs += f"Document {i+1}:\n{doc.page_content}\n\n"
    return formatted_docs

rag_chain = (
    {"context": retriever, "question": RunnablePassThrough(), "retrieved_docs": format_retrieved_docs} | rag_prompt | llm
)

In [ ]:
@tool
def rag_gpt_tool(question: str) -> str:
    retrieved_docs = retriever(question)
    formatted_docs = format_retrieved_docs(retrieved_docs)
    response = rag_chain.run({"question": question, "retrieved_docs": formatted_docs})
    return response

tools = [rag_gpt_tool]
agent = create_react_agent(tools, llm, verbose=True)

# Document Generation Agent with LlamaIndex

In [ ]:
from tavily import AsyncTavilyClient
from llama_index.core import FunctionTool

TAVIL_API_KEY = os.getenv("TAVIL_API_KEY")

async def web_search(query: str) -> str:
    client = AsyncTavilyClient(api_key=TAVIL_API_KEY)
    result = await client.search(query)
    return str(result)


web_search_tool = FunctionTool.from_defaults(
    func=web_search,
    name="web_search",
    description="Use this tool to search the web for up-to-date information.",
)


In [ ]:
llm = Anthropic(model=model)
Settings.llm = llm
agent = FunctionAgent(
    tools=[web_search_tool],
    llm=llm,
    verbose=True,
    system_prompt="You are a helpful assistant that can use tools to answer questions."
)

In [ ]:
ctx = Context(agent)

q1 = (
    "What is the current stock price of Apple Inc?"
    "Find the answer by using the web_search tool to search for 'current stock price of Apple Inc' and return the result."
)

hanlder = agent.run(q1, ctx=ctx)

async for event in hanlder.stream_events():
    if event.type == "tool_call":
        print(f"Tool called: {event.tool_call.name}")
        print(f"Arguments: {event.tool_call.arguments}")
    elif event.type == "tool_response":
        print(f"Tool response: {event.tool_response}")
    elif event.type == "agent_thought":
        print(f"Agent thought: {event.agent_thought.thought}")
    elif event.type == "message":
        print(f"Agent message: {event.message.content}")

# Building an agent with Vectara

In [ ]:
import json
import requests
import os

In [ ]:
url = "https://api.tavily.com/v1/search"

payload = {}

files = [ ('file', open('query.txt','rb')) ]
headers = {
    'accept': 'application/json',
    'Tavil-Key': os.getenv("TAVIL_API_KEY")
}   


tool_configurations = {
    "web_search": {
        "type": "function",
        "name": "web_search",
        "description": "Use this tool to search the web for up-to-date information.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to find relevant information on the web.",
                },
            },
            "required": ["query"],
        },
    }
}

In [ ]:
agent_key = "my-agent-key"
create_agent_obj = {
    "agent_key": agent_key,
    "name": "Web Search Agent",
    "description": "An agent that can search the web for up-to-date information using the web_search tool.",
    "tool_configurations": tool_configurations,
}
url = "https://api.tavily.com/v1/agents"
headers = {
    "Content-Type": "application/json",
    "Tavil-Key": os.getenv("TAVIL_API_KEY")
}

response = requests.post(url, headers=headers, data=json.dumps(create_agent_obj))
print(response.status_code)

# Build a multi-agent system with CrewAI

In [ ]:
import os
from crewai import Agent, Task, Crew, Process

researcher = Agent(
    name="Researcher",
    description="An agent that researches information on the web using the web_search tool.",
    tools=["web_search"],
)

writer = Agent(
    name="Writer",
    description="An agent that writes reports based on information provided by the Researcher agent.",
)

In [ ]:
resercher_task = Task(
    name="Research Task",
    description="Use the web_search tool to find information about the current stock price of Apple Inc.",
    agent=researcher,
)

writer_task = Task(
    name="Writing Task",
    description="Write a report about the current stock price of Apple Inc. based on the information provided by the Researcher agent.",
    agent=writer,
)

In [ ]:
ai_trends_crew = Crew(
    name="AI Trends Crew",
    description="A crew that researches and writes reports about current trends in AI.",
    agents=[researcher, writer],
    tasks=[resercher_task, writer_task],
)

result = ai_trends_crew.kickoff()
print(result)